**Chicago Ventra — PS1: Predictive Failure (7-Day Binary)**

---

Problem Statement: Predict whether a fare-collection device (TVM, GATE, VALIDATOR) will experience a failure event within the next 7 days, based on daily telemetry aggregates.

- Grain: (DEVICE_ID, transit_day)  — 8,924 rows
- Target: will_fail_7d (binary 0/1)
- Models: LightGBM · XGBoost · CatBoost · RandomForest · LogisticRegression → stacked with LR meta-learner
- Split: TimeSeriesSplit (n_splits=5) sorted on transit_day — no shuffle
- Evaluation: ROC-AUC, F1, Precision-Recall curve, Confusion Matrix
- Explainability: SHAP TreeExplainer per model
- Gold table: device_ps1_daily.parquet

Scope: READER (SCR, 179 devices) excluded — 0 OOS events confirmed in device_event_enriched (2026-06-19).
RTL (2,176 Retail Terminals) remain in TVM category; excluded from gold SQL scope via device_ps1_daily__create.sql filter.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score, roc_curve
)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import shap

sns.set_theme(style='whitegrid', palette='tab10')
SEED = 42
print('All imports OK')

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────
GOLD_PATH = r'D:\Sathish\Chicago_Synthetic data\chicago_oracle_real_data\gold'

df = pd.read_parquet(f'{GOLD_PATH}\\device_ps1_daily.parquet')
df['transit_day'] = pd.to_datetime(df['transit_day'])
df = df.sort_values(['DEVICE_ID', 'transit_day']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["transit_day"].min().date()} → {df["transit_day"].max().date()}')
print(f'Devices: {df["DEVICE_ID"].nunique()}')
df.head(3)

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
print('=== Dtypes ===')
print(df.dtypes)
print('\n=== Nulls (top 10) ===')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print('\n=== Target Distribution ===')
vc = df['will_fail_7d'].value_counts()
print(vc)
print(f'\nClass imbalance ratio: {vc[1]/vc[0]:.2f}  (positives per negative)')

In [ ]:
# Target distribution plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vc = df['will_fail_7d'].value_counts().sort_index()
axes[0].bar(['No Failure (0)', 'Will Fail (1)'], vc.values, color=['steelblue', 'tomato'])
axes[0].set_title('Target: will_fail_7d Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Device-type breakdown
dt_counts = df.groupby('mars_device_category')['will_fail_7d'].value_counts().unstack(fill_value=0)
dt_counts.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'])
axes[1].set_title('Failures by Device Category')
axes[1].set_xlabel('Device Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(['No Failure', 'Will Fail'])

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features
FEATURE_COLS = [
    # Same-day event counters (gold column names from device_ps1_daily)
    'critical_events', 'oos_event_count', 'bhu_events', 'chu_events',
    'printer_events', 'gate_mech_events', 'csc_reader_events', 'comms_events',
    'bankcard_events', 'system_events', 'scrst_events', 'event_count',
    # Rolling window features (pre-computed in gold SQL)
    'critical_events_7d', 'critical_events_30d', 'oos_events_7d', 'events_7d',
    # Operational KPIs — actual gold column names (tap_reject_rate_pct, availability_pct_7d)
    # kpi_availability_pct and kpi_fault_count removed from S09 (BUG 18/19 fix 2026-06-19)
    'tap_reject_rate_pct', 'availability_pct_7d', 'outage_count',
]

# ── S21 use_revenue_daily additions (2026-06-25) ─────────────────────────────
# Revenue features from EDW.USE_TRANSACTION — cast bool flag to int before model
if "use_revenue_decline_flag" in df.columns:
    df["use_revenue_decline_flag"] = df["use_revenue_decline_flag"].astype(int)
REVENUE_FEATURES = [c for c in [
    'use_txn_count', 'use_priced_txn_count', 'use_revenue_cents',
    'use_net_revenue_cents', 'use_revenue_active_hours',
    'use_revenue_7d_avg', 'use_revenue_decline_flag',
] if c in df.columns]

# ── S22 read_tap_daily additions (2026-06-25) ─────────────────────────────────
READ_TAP_FEATURES = [c for c in [
    'daily_read_count', 'unique_tokens', 'approved_read_count',
    'rejected_read_count', 'reject_rate_pct', 'read_active_hours',
    'transfer_tap_count',
] if c in df.columns]

FEATURE_COLS = FEATURE_COLS + REVENUE_FEATURES + READ_TAP_FEATURES
print(f"Total features: {len(FEATURE_COLS)} (base + {len(REVENUE_FEATURES)} revenue + {len(READ_TAP_FEATURES)} read-tap)")

corr_df = df[FEATURE_COLS + ['will_fail_7d']].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr_df, cmap='RdBu_r', center=0, annot=False, fmt='.2f', linewidths=0.3)
plt.title('Feature Correlation Matrix (PS1)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
df_fe = df.copy()

# Encode device category
le_cat = LabelEncoder()
df_fe['device_cat_enc'] = le_cat.fit_transform(df_fe['mars_device_category'])

# Rolling aggregates per device (lag features must be sorted by transit_day)
df_fe = df_fe.sort_values(['DEVICE_ID', 'transit_day']).reset_index(drop=True)
df_fe['critical_rolling_3d'] = (
    df_fe.groupby('DEVICE_ID')['critical_events']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
df_fe['oos_rolling_3d'] = (
    df_fe.groupby('DEVICE_ID')['oos_event_count']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
df_fe['total_events_rolling_7d'] = (
    df_fe.groupby('DEVICE_ID')['event_count']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
# Interaction: missed KPI targets × unavailability
# Uses kpi_targets_missed (gold column) — kpi_fault_count was removed from S09 (BUG 18/19)
df_fe['severity_x_unavail'] = (
    df_fe['kpi_targets_missed'].fillna(0) *
    (100 - df_fe['availability_pct_7d'].fillna(100))
)
# Event density: events per unit availability
# Uses availability_pct_7d from rolling window — kpi_availability_pct removed from gold
df_fe['event_density'] = (
    df_fe['event_count'] /
    (df_fe['availability_pct_7d'].fillna(100).clip(1, 100) / 100)
)

FEATURE_COLS_EXT = FEATURE_COLS + [
    'device_cat_enc', 'critical_rolling_3d', 'oos_rolling_3d',
    'total_events_rolling_7d', 'severity_x_unavail', 'event_density'
]

# Fill any remaining NaNs with 0
df_fe[FEATURE_COLS_EXT] = df_fe[FEATURE_COLS_EXT].fillna(0)

print(f'Feature count: {len(FEATURE_COLS_EXT)}')
print('Sample engineered features:')
df_fe[['DEVICE_ID', 'transit_day', 'critical_rolling_3d', 'severity_x_unavail', 'event_density']].head(5)

In [ ]:
# ── Train / Test Split (TimeSeriesSplit) ──────────────────────────────────────
# Sort globally by transit_day for time-ordered splitting
df_model = df_fe.sort_values('transit_day').reset_index(drop=True)

X = df_model[FEATURE_COLS_EXT].values
y = df_model['will_fail_7d'].values

tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(X, y))

# Use final fold for hold-out evaluation
train_idx, test_idx = splits[-1]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'Train size: {len(X_train)}  Test size: {len(X_test)}')
print(f'Train date range: {df_model["transit_day"].iloc[train_idx[0]].date()} → {df_model["transit_day"].iloc[train_idx[-1]].date()}')
print(f'Test  date range: {df_model["transit_day"].iloc[test_idx[0]].date()} → {df_model["transit_day"].iloc[test_idx[-1]].date()}')
print(f'Train pos rate: {y_train.mean():.3f}  Test pos rate: {y_test.mean():.3f}')

In [ ]:
# ── Model Training ────────────────────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos:.3f}')

lgbm_model = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    scale_pos_weight=scale_pos, random_state=SEED, n_jobs=-1, verbose=-1
)
xgb_model = xgb.XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    scale_pos_weight=scale_pos, random_state=SEED,
    eval_metric='logloss', verbosity=0
)
cb_model = CatBoostClassifier(
    iterations=400, learning_rate=0.05, depth=6,
    scale_pos_weight=scale_pos, random_seed=SEED, verbose=0
)
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)
lr_model = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=SEED
)

base_estimators = [
    ('lgbm', lgbm_model),
    ('xgb',  xgb_model),
    ('cb',   cb_model),
    ('rf',   rf_model),
    ('lr',   lr_model)
]

# Stacking classifier with LR meta-learner
stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3,
    stack_method='predict_proba',
    n_jobs=-1
)

print('Training stacking ensemble...')
stack_model.fit(X_train, y_train)
print('Training complete.')

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
# Also train individual models for comparison
individual_models = {}
for name, model in base_estimators:
    model.fit(X_train, y_train)
    individual_models[name] = model

all_models = {**individual_models, 'stack': stack_model}

results = []
for name, model in all_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    results.append({
        'Model': name.upper(),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'F1': f1_score(y_test, y_pred),
        'Avg Precision': average_precision_score(y_test, y_prob),
        'y_prob': y_prob,
        'y_pred': y_pred
    })

results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('y_prob', 'y_pred')} for r in results])
results_df = results_df.sort_values('ROC-AUC', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# ROC curves + PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in results:
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    auc = roc_auc_score(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{r['Model']} (AUC={auc:.3f})")
axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves — PS1 Predictive Failure')
axes[0].legend(fontsize=8)

for r in results:
    prec, rec, _ = precision_recall_curve(y_test, r['y_prob'])
    ap = average_precision_score(y_test, r['y_prob'])
    axes[1].plot(rec, prec, label=f"{r['Model']} (AP={ap:.3f})")
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves — PS1')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for top 3 models by ROC-AUC
top3 = results_df.head(3)['Model'].str.lower().tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, top3):
    r = next(x for x in results if x['Model'].lower() == name)
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Fail', 'Will Fail'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{r['Model']} Confusion Matrix")
plt.tight_layout()
plt.show()

# Detailed classification report for stacking model
print('\n=== Stacking Model Classification Report ===')
stack_r = next(x for x in results if x['Model'] == 'STACK')
print(classification_report(y_test, stack_r['y_pred'], target_names=['No Failure', 'Will Fail']))

In [ ]:
# ── SHAP Feature Importance ───────────────────────────────────────────────────
# Use LightGBM (best single-model baseline typically)
print('Computing SHAP values for LightGBM...')
explainer_lgbm = shap.TreeExplainer(individual_models['lgbm'])
shap_values_lgbm = explainer_lgbm.shap_values(X_test)

# For binary classification shap_values may be list [neg, pos] or single array
if isinstance(shap_values_lgbm, list):
    sv = shap_values_lgbm[1]
else:
    sv = shap_values_lgbm

feature_names = FEATURE_COLS_EXT

# Summary bar plot
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, feature_names=feature_names,
                  plot_type='bar', show=False, max_display=20)
plt.title('SHAP Feature Importance — LightGBM (PS1)')
plt.tight_layout()
plt.show()

# Beeswarm plot
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, feature_names=feature_names,
                  plot_type='dot', show=False, max_display=15)
plt.title('SHAP Beeswarm — LightGBM (PS1)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP for XGBoost
print('Computing SHAP values for XGBoost...')
explainer_xgb = shap.TreeExplainer(individual_models['xgb'])
shap_values_xgb = explainer_xgb.shap_values(X_test)
if isinstance(shap_values_xgb, list):
    sv_xgb = shap_values_xgb[1]
else:
    sv_xgb = shap_values_xgb

plt.figure(figsize=(10, 7))
shap.summary_plot(sv_xgb, X_test, feature_names=feature_names,
                  plot_type='bar', show=False, max_display=20)
plt.title('SHAP Feature Importance — XGBoost (PS1)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-Device-Type Analysis ──────────────────────────────────────────────────
test_df = df_model.iloc[test_idx].copy().reset_index(drop=True)
test_df['lgbm_prob'] = individual_models['lgbm'].predict_proba(X_test)[:, 1]
test_df['stack_prob'] = stack_model.predict_proba(X_test)[:, 1]
test_df['stack_pred'] = stack_model.predict(X_test)

device_results = []
for cat in sorted(test_df['mars_device_category'].unique()):
    sub = test_df[test_df['mars_device_category'] == cat]
    if len(sub) < 10 or sub['will_fail_7d'].nunique() < 2:
        continue
    auc = roc_auc_score(sub['will_fail_7d'], sub['stack_prob'])
    f1  = f1_score(sub['will_fail_7d'], sub['stack_pred'])
    pos_rate = sub['will_fail_7d'].mean()
    device_results.append({'Device': cat, 'N': len(sub),
                            'Pos_Rate': round(pos_rate, 3),
                            'ROC-AUC': round(auc, 3), 'F1': round(f1, 3)})

dev_df = pd.DataFrame(device_results).sort_values('ROC-AUC', ascending=False)
print('Per-Device-Type Results (Stacking Model):')
print(dev_df.to_string(index=False))

In [ ]:
# Per-device ROC-AUC bar chart
plt.figure(figsize=(10, 4))
colors = ['green' if v >= 0.7 else 'orange' if v >= 0.5 else 'red' for v in dev_df['ROC-AUC']]
plt.bar(dev_df['Device'], dev_df['ROC-AUC'], color=colors)
plt.axhline(0.5, color='gray', linestyle='--', label='Random baseline')
plt.axhline(0.7, color='green', linestyle='--', alpha=0.4, label='Good threshold (0.70)')
plt.ylim(0.3, 1.0)
plt.title('ROC-AUC by Device Category — PS1 Stacking Model')
plt.xlabel('Device Category')
plt.ylabel('ROC-AUC')
plt.legend()
plt.xticks(rotation=30)
for i, row in dev_df.reset_index().iterrows():
    plt.text(i, row['ROC-AUC'] + 0.01, f"{row['ROC-AUC']:.3f}", ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Risk score distribution by device category
plt.figure(figsize=(11, 5))
for cat in sorted(test_df['mars_device_category'].unique()):
    sub = test_df[test_df['mars_device_category'] == cat]
    sns.kdeplot(sub['stack_prob'], label=cat, fill=False)
plt.axvline(0.5, color='red', linestyle='--', label='Decision threshold (0.5)')
plt.title('Predicted Failure Probability Distribution by Device Category')
plt.xlabel('Predicted P(will_fail_7d=1)')
plt.ylabel('Density')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation AUC scores across all 5 folds
cv_aucs = {'Model': [], 'Fold': [], 'ROC-AUC': []}
cv_models = {'LightGBM': lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                              scale_pos_weight=scale_pos,
                                              random_state=SEED, verbose=-1),
             'XGBoost':  xgb.XGBClassifier(n_estimators=200, learning_rate=0.05,
                                            scale_pos_weight=scale_pos,
                                            random_state=SEED, verbosity=0),
             'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                                    random_state=SEED, n_jobs=-1)}
for mname, mdl in cv_models.items():
    for fold_i, (tr, te) in enumerate(tscv.split(X, y)):
        mdl_clone = type(mdl)(**mdl.get_params())
        mdl_clone.fit(X[tr], y[tr])
        prob = mdl_clone.predict_proba(X[te])[:, 1]
        if len(np.unique(y[te])) > 1:
            cv_aucs['Model'].append(mname)
            cv_aucs['Fold'].append(fold_i + 1)
            cv_aucs['ROC-AUC'].append(roc_auc_score(y[te], prob))

cv_df = pd.DataFrame(cv_aucs)
plt.figure(figsize=(8, 4))
sns.boxplot(data=cv_df, x='Model', y='ROC-AUC', palette='Set2')
plt.title('5-Fold TimeSeriesSplit ROC-AUC Distribution')
plt.tight_layout()
plt.show()
print(cv_df.groupby('Model')['ROC-AUC'].agg(['mean', 'std']).round(4))

**Key Findings — PS1 Predictive Failure**

**Model Performance Summary**
- **Best model:** Stacking ensemble (LightGBM + XGBoost + CatBoost + RF + LR → LR meta-learner)
- **Evaluation metric:** ROC-AUC on hold-out time-ordered test split
- **Class imbalance:** ~9:1 positive-to-negative ratio (`will_fail_7d=1` is dominant) — handled via `scale_pos_weight` and `class_weight='balanced'`

**Top Predictive Features (SHAP)**
1. `critical_events` — same-day critical event count is the strongest single predictor
2. `critical_events_7d` / `critical_events_30d` — historical critical event accumulation over 7- and 30-day rolling windows
3. `availability_pct_7d` — low 7-day rolling availability strongly precedes failure
4. `oos_event_count` / `oos_events_7d` — out-of-service events are a leading indicator
5. `event_count` — total daily event volume correlates with device stress
6. `tap_reject_rate_pct` — elevated reject rates indicate degraded state

**Device-Type Patterns**
- TVM and GATE devices show different failure signatures vs. bus validators (VALIDATOR)
- READER (SCR) excluded from scope: 0 OOS events confirmed (2026-06-19)
- Some device categories have insufficient negative-class samples for reliable per-type ROC-AUC

**Limitations**
- Real Oracle bronze data: gold rows = 0 for 50-row samples (DEVICE_KEY non-overlap between dim_device and device_event_enriched). Full Oracle data will populate gold rows.
- No weather / external event features included
- `incident_duration_min` not available at prediction time (future leakage)
- TimeSeriesSplit does not group by device, so some fold leakage is possible if devices appear in both train and test windows
- `kpi_targets_missed` coverage ≈10% (kpi_daily sparsity); `metric_txn_count` coverage ≈26% (metric_daily 7-month gap pre-Jun 2026)